In [ ]:
import torch.optim
import torch.utils.data
import torchvision.transforms as transforms
# from datasets import *
# from utils import *
import torch.nn.functional as F
from tqdm import tqdm
import argparse
import time
import os
import json
from eval_func.bleu.bleu import Bleu
from eval_func.rouge.rouge import Rouge
from eval_func.cider.cider import Cider
from eval_func.meteor.meteor import Meteor

In [ ]:
def get_key(dict_, value):
  return [k for k, v in dict_.items() if v == value]

def get_eval_score(references, hypotheses):
    scorers = [
        (Bleu(4), ["Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4"]),
        (Meteor(), "METEOR"),
        (Rouge(), "ROUGE_L"),
        (Cider(), "CIDEr")
    ]

    hypo = [[' '.join(hypo)] for hypo in [[str(x) for x in hypo] for hypo in hypotheses]]
    ref = [[' '.join(reft) for reft in reftmp] for reftmp in
           [[[str(x) for x in reft] for reft in reftmp] for reftmp in references]]

    score = []
    method = []
    for scorer, method_i in scorers:
        score_i, scores_i = scorer.compute_score(ref, hypo)
        score.extend(score_i) if isinstance(score_i, list) else score.append(score_i)
        method.extend(method_i) if isinstance(method_i, list) else method.append(method_i)
        print("{} {}".format(method_i, score_i))
    score_dict = dict(zip(method, score))

    return score_dict


In [ ]:
# Load word map (word2ix)
word_map_file = os.path.join(r'/root/autodl-tmp/Blip2CC/WORDMAP_LEVIR_CC_5_cap_per_img_5_min_word_freq.json')
with open(word_map_file, 'r') as f:
    word_map = json.load(f)

rev_word_map = {v: k for k, v in word_map.items()}
vocab_size = len(word_map)

In [ ]:
reference_path = r'/root/autodl-tmp/Blip2CC/.cache/coco/annotations/coco_karpathy_test.json'
with open(reference_path, 'r') as j:
    gt_raw = json.load(j)

references_all = []
references_no_changed = []
references_changed = []

for item in gt_raw:
    caps = [x.replace(" .", "").strip() for x in item['captions']]
    caps = [x.split(' ') for x in caps]
    caps_enc = []
    for j, c in enumerate(caps):
        enc_c = [word_map.get(word, word_map['<unk>']) for word in c]
        caps_enc.append(enc_c)
    caps = list(
                map(lambda c: [w for w in c if w not in {word_map['<start>'], word_map['<end>'], word_map['<pad>']}],
                    caps_enc))
    if item['changeflag'] == 0:
        references_no_changed.append(caps_enc)
    else:
        references_changed.append(caps_enc)
    references_all.append(caps_enc)

In [ ]:
path = r'/root/autodl-tmp/Blip2CC/output/BLIP2/Caption_coco_opt2.7b/20250320205/test_epochbest.json'
results = []
with open(path, 'r') as f:
    data = json.load(f)
results = [cap['caption'].replace(".", "") for cap in data]
# results = [cap.replace(".", "") for cap in data]
results = [x.split(' ') for x in results]

hypotheses_all = []
hypotheses_no_changed = []
hypotheses_changed = []

for gt, result in zip(gt_raw, results):
    answer = [word_map.get(word, word_map['<unk>']) for word in result]

    if gt['changeflag'] == 0:
        hypotheses_no_changed.append(answer)
    else:
        hypotheses_changed.append(answer)
    
    hypotheses_all.append(answer)


In [ ]:
assert len(references_all) == len(hypotheses_all)
assert len(references_no_changed) == len(hypotheses_no_changed)
assert len(references_changed) == len(hypotheses_changed)

metrics_all = get_eval_score(references_all, hypotheses_all)
metrics_no_changed = get_eval_score(references_no_changed, hypotheses_no_changed)
metrics_changed = get_eval_score(references_changed, hypotheses_changed)

print("====== no changed results ======")
print("BLEU-1 {} BLEU-2 {} BLEU-3 {} BLEU-4 {} METEOR {} ROUGE_L {} CIDEr {}".format
        (metrics_no_changed["Bleu_1"], metrics_no_changed["Bleu_2"], metrics_no_changed["Bleu_3"], metrics_no_changed["Bleu_4"],
        metrics_no_changed["METEOR"], metrics_no_changed["ROUGE_L"], metrics_no_changed["CIDEr"]))
print("\n")
print("====== changed results ======")
print("BLEU-1 {} BLEU-2 {} BLEU-3 {} BLEU-4 {} METEOR {} ROUGE_L {} CIDEr {}".format
        (metrics_changed["Bleu_1"], metrics_changed["Bleu_2"], metrics_changed["Bleu_3"], metrics_changed["Bleu_4"],
        metrics_changed["METEOR"], metrics_changed["ROUGE_L"], metrics_changed["CIDEr"]))

print("\n")
print("====== all results ======")
print("BLEU-1 {} BLEU-2 {} BLEU-3 {} BLEU-4 {} METEOR {} ROUGE_L {} CIDEr {}".format
        (metrics_all["Bleu_1"], metrics_all["Bleu_2"], metrics_all["Bleu_3"], metrics_all["Bleu_4"],
        metrics_all["METEOR"], metrics_all["ROUGE_L"], metrics_all["CIDEr"]))

In [ ]:
references = list()
hypotheses = list()
change_references = list()
change_hypotheses = list()
nochange_references = list()
nochange_hypotheses = list()
change_acc=0
nochange_acc=0

with torch.no_grad():
    for i, (image_pairs, caps, caplens, allcaps) in enumerate(
            tqdm(loader, desc=args.Split + " EVALUATING AT BEAM SIZE " + str(beam_size))):
        # 5 image is the same when "shuffle=False" of the dataloader
        if (i + 1) % 5 != 0:
            continue
       
        if (len(complete_seqs_scores) > 0):
            indices = complete_seqs_scores.index(max(complete_seqs_scores))
            seq = complete_seqs[indices]
            # References
            img_caps = allcaps[0].tolist()
            img_captions = list(
                map(lambda c: [w for w in c if w not in {word_map['<start>'], word_map['<end>'], word_map['<pad>']}],
                    img_caps))  # remove <start> and pads

            references.append(img_captions)
            # Hypotheses
            new_sent = [w for w in seq if w not in {word_map['<start>'], word_map['<end>'], word_map['<pad>']}]
            hypotheses.append(new_sent)
            assert len(references) == len(hypotheses)

            # # 判断有没有变化
            nochange_list = ["the scene is the same as before ", "there is no difference ",
                                "the two scenes seem identical ", "no change has occurred ",
                                "almost nothing has changed "]
            ref_sentence = img_captions[1]
            ref_line_repo = ""
            for ref_word_idx in ref_sentence:
                ref_word = get_key(word_map, ref_word_idx)
                ref_line_repo += ref_word[0] + " "

            hyp_sentence = new_sent
            hyp_line_repo = ""
            for hyp_word_idx in hyp_sentence:
                hyp_word = get_key(word_map, hyp_word_idx)
                hyp_line_repo += hyp_word[0] + " "
            # 对于变化图像对
            if ref_line_repo not in nochange_list:
                change_references.append(img_captions)
                change_hypotheses.append(new_sent)
                if hyp_line_repo not in nochange_list:
                    change_acc = change_acc+1
            else:
                nochange_references.append(img_captions)
                nochange_hypotheses.append(new_sent)
                if hyp_line_repo in nochange_list:
                    nochange_acc = nochange_acc+1

    # captions
    # save_captions(args, word_map, hypotheses, references)

print('len(nochange_references):', len(nochange_references))
print('len(change_references):', len(change_references))
# Calculate BLEU1~4, METEOR, ROUGE_L, CIDEr scores
if len(nochange_references)>0:
    print('nochange_metric:')
    nochange_metric = get_eval_score(nochange_references, nochange_hypotheses)
    print("nochange_acc:", nochange_acc / len(nochange_references))
if len(change_references)>0:
    print('change_metric:')
    change_metric = get_eval_score(change_references, change_hypotheses)
    print("change_acc:", change_acc / len(change_references))
print(".......................................................")
metrics = get_eval_score(references, hypotheses)